In [ ]:
%pip install "pandas>=2.0.0" "openai>=2.20.0" "python-dotenv>=1.0.0"

In [ ]:
import os
import json
from pprint import pprint

import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

# 합성 데이터

In [ ]:
# 1. load_dotenv() 함수를 사용하여 .env 파일을 로드하세요
load_dotenv()

# 2. os.getenv() 함수를 사용하여 UPSTAGE_API_KEY를 가져오세요
UPSTAGE_API_KEY = os.getenv('UPSTAGE_API_KEY')

# 정상적으로 잘 작동하는지 확인해봅시다.
if UPSTAGE_API_KEY:
    print("Success API Key Setting!")
else:
    print("Failed to load API Key. Please check your .env file.")

In [ ]:
# 1. OpenAI 클라이언트 생성
client = OpenAI(
    api_key=UPSTAGE_API_KEY,
    base_url="https://api.upstage.ai/v1"
)

# 2. chat_completion 함수를 완성
def chat_completion(prompt: str,
                    system_prompt: str = None,
                    model: str = "solar-pro3") -> str:
    """
    LLM을 호출하여 응답을 반환하는 함수

    Args:
        prompt: 사용자 메시지
        system_prompt: 시스템 메시지 (선택)
        model: 사용할 모델명

    Returns:
        LLM의 응답 텍스트
    """
    messages = []

    # 시스템 프롬프트는 LLM의 동작을 제어하는 프롬프트로
    # 대화의 스타일, 성격, 제약 조건 등을 결정합니다.
    # LLM이 어떤 방식으로 답변해야 하는지를 미리 정해주는 지침으로 이해하면 됩니다.
    # 필수는 아니기 때문에 `if`문으로 처리하였습니다.
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

# 잘 작동하는지 테스트해볼까요?
test_response = chat_completion("안녕하세요! 간단한 인사말을 해주세요.")
print("테스트 응답:")
print(test_response) 

## Prompt Engineering

In [ ]:
# 공통 시스템 프롬프트
SYSTEM_PROMPT = """당신은 세상의 모든 영화를 꿰뚫고 있는 영화 전문가 '시네마스터'입니다.
사용자의 요청에 맞춰 영화를 추천하는 역할을 맡고 있습니다.
추천할 때는 반드시 영화 제목, 개봉 연도, 그리고 추천 이유를 포함해야 합니다."""

user_query = "스릴러 영화를 추천해줘"

# 1. Zero-shot 프롬프트 `zero_shot_prompt`를 작성하세요 (예시 없이 직접 지시)
zero_shot_prompt = f'{user_query}'

print("=" * 50)
print("Zero-shot 프롬프팅 결과:")
print("=" * 50)
zero_shot_result = chat_completion(zero_shot_prompt, SYSTEM_PROMPT)
print(zero_shot_result)
print() 

In [ ]:
# 2. Few-shot 프롬프트 `few_shot_prompt`를 작성하세요 (2-3개의 예시 포함)
few_shot_prompt = f"""다음은 영화 추천 예시입니다:

질문: 경제 영화를 추천해줘
답변: 영화 제목: 빅쇼트 (The Big Short)
개봉 연도: 2015년
추천 이유: 제2차 세계 대전 이후 미국 최대, 최악의 금융 위기를 쉽게 이해할 수 있는 구성과 더불어, 뛰어난 편집, 연출이 돋보이는 영화입니다.

질문: 코미디 영화를 추천해줘
답변: 영화 제목: 행오버 (The Hangover)
개봉 연도: 2009년
추천 이유: 총각파티 후 기억을 잃은 친구들의 황당한 모험을 그린 코미디로, 예측 불가능한 전개와 유쾌한 유머가 가득합니다.

질문: {user_query}
답변: """

print("=" * 50)
print("Few-shot 프롬프팅 결과:")
print("=" * 50)
few_shot_result = chat_completion(few_shot_prompt, SYSTEM_PROMPT)
pprint(few_shot_result)
print() 

In [ ]:
# 3. CoT 프롬프트 `cot_prompt`를 작성하세요 (단계별 추론 유도)
cot_prompt = f"""{user_query}

영화를 추천하기 전에 다음 단계를 따라 생각해주세요:

1단계: 스릴러 장르의 핵심 요소가 무엇인지 정의합니다 (긴장감, 반전, 서스펜스 등)
2단계: 이 요소들을 잘 갖춘 대표적인 스릴러 영화들을 떠올립니다
3단계: 그 중에서 가장 추천할 만한 영화 1개를 선택하고 이유를 설명합니다

위 단계를 따라 추론 과정을 보여주고, 최종 추천을 해주세요."""

print("=" * 50)
print("Chain-of-Thought 프롬프팅 결과:")
print("=" * 50)
cot_result = chat_completion(cot_prompt, SYSTEM_PROMPT)
print(cot_result)
print()

## 구조화된 합성 데이터 생성

In [ ]:
def json_parsing(output_text: str) -> dict:
    """
    LLM 응답에서 JSON 부분을 추출하여 딕셔너리로 변환

    Args:
        output_text: LLM의 응답 텍스트

    Returns:
        파싱된 딕셔너리
    """
    try:
        # ```json ... ``` 형식에서 JSON 추출
        if "```json" in output_text:
            output_text = output_text[output_text.index("```json") + len("```json"):].strip()
            output_text = output_text[:output_text.index("```")].strip()
        return json.loads(output_text)
    except (json.JSONDecodeError, ValueError) as e:
        print(f"JSON 파싱 오류: {e}")
        return None

In [ ]:
# 1. 구조화된 출력을 위한 시스템 프롬프트를 작성하세요
STRUCTURED_GENERATOR_SYSTEM_PROMPT = """당신은 세상의 모든 영화를 꿰뚫고 있는 영화 전문가 '시네마스터'입니다.
사용자의 요청에 맞춰 영화를 추천하는 역할을 맡고 있습니다. 영화는 반드시 하나만 추천합니다.

## 1. 입력 형식
[추천받고자 하는 영화 장르]

## 2. 작업 지시
- 요청된 장르에 가장 적합한 영화 1개를 추천합니다.
- 추천 이유는 구체적이고 설득력 있게 작성합니다.
- 친근하고 유머러스한 말투로 설명합니다.

## 3. 출력 형식
- 출력 형식은 다음 포맷을 따릅니다.
```json
{
    "movie_name": [영화 이름],
    "year": [개봉 연도],
    "genre": [장르],
    "reason": [추천 이유]
}
```
"""

# 2. 1에서 만든 프롬프트를 활용하여 답변을 받고,
# 답변을 JSON으로 파싱하여 `dict` 형태로 반환하도록 함수를 구성해주세요.
def generate_movie_recommendation(genre: str, temperature: float = 1.0) -> dict:
    """
    특정 장르에 대한 영화 추천 데이터를 생성

    Args:
        genre: 영화 장르
        temperature: 다양성 조절 파라미터 (0.0~2.0)

    Returns:
        구조화된 영화 추천 데이터
    """
    response = client.chat.completions.create(
        model='solar-pro3',
        messages=[
            {'role': 'system', 'content': STRUCTURED_GENERATOR_SYSTEM_PROMPT},
            {'role': 'user', 'content': f'{genre} 영화를 추천해줘'}
        ]
    )
    output = response.choices[0].message.content
    return json_parsing(output)

# 3. 3가지 다른 장르에 대해 합성 데이터를 생성하고 `synthetic_data`에 저장해주세요.
genres = ["공포", "SF", "액션"]
synthetic_data = []

for genre in genres:
    print(f'\n{genre} 장르 데이터 생성 중...')
    data = generate_movie_recommendation(genre, temperature=1.0)
    if data:
        synthetic_data.append(data)
        print('생성 완료: ', end='')
        pprint(data)

# 생성된 데이터 확인
print("\n" + "=" * 50)
print("생성된 합성 데이터:")
print("=" * 50)
for i, data in enumerate(synthetic_data, 1):
    print(f"\n[{i}] ", end=""); pprint(data)

In [ ]:
synthetic_data[0]['movie_name']